# Aplicaciones de Transformers en Procesamiento de Lenguaje Natural
Este cuaderno explora distintas aplicaciones de los modelos Transformers en tareas de procesamiento de lenguaje natural (PLN), utilizando ejemplos prácticos y explicaciones teóricas.

## 1. Definir el texto de ejemplo

In [ ]:
texto = """Querido MercadoLibre, la semana pasada pedí una figura de acción de Optimus Prime desde su tienda online.
Para mi sorpresa, cuando abrí el paquete, descubrí horrorizado que me habían enviado una figura de Megatron.
Como fan de los Autobots, espero que entiendan mi decepción. Solicito un cambio urgente del producto."""

## 2. Clasificación de texto con Transformers

In [ ]:
from transformers import pipeline

In [ ]:
# Modelo de clasificación de texto (sentimiento) en español
classifier = pipeline("text-classification", model="pysentimiento/robertuito-sentiment-analysis")

In [ ]:
import pandas as pd
outputs = classifier(texto)
pd.DataFrame(outputs)

## 3. Reconocimiento de entidades nombradas (NER)

In [ ]:
ner_tagger = pipeline("ner", model="mrm8488/bert-spanish-cased-finetuned-ner", aggregation_strategy="simple")

In [ ]:
outputs = ner_tagger(texto)
pd.DataFrame(outputs)

In [ ]:
texto_2 = "Lionel Messi nació en Rosario, jugó en el FC Barcelona y ahora vive en Miami."
resultado = ner_tagger(texto_2)
pd.DataFrame(resultado)

## 4. Respuesta a preguntas basada en contexto

In [ ]:
reader = pipeline("question-answering", model="PlanTL-GOB-ES/roberta-large-bne-sqac")

In [ ]:
question = "¿Qué quiere el cliente?"
outputs = reader(question=question, context=texto)
pd.DataFrame([outputs])

## 5. Resumen automático de texto

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
resumidor = pipeline("summarization", model=model, tokenizer=tokenizer)

In [ ]:
resumen = resumidor(texto, max_length=80, min_length=20, do_sample=False)
print(resumen[0]['summary_text'])

## 6. Traducción automática del español a inglés

In [ ]:
translator = pipeline("translation_es_to_en", model="Helsinki-NLP/opus-mt-es-en")
outputs = translator(texto, clean_up_tokenization_spaces=True, min_length=130)
print(outputs[0]['translation_text'])

## 7. Generación automática de texto (respuesta de servicio al cliente)

In [ ]:
generador = pipeline("text-generation", model="datificate/gpt2-small-spanish")
respuesta_inicial = "Estimado Bumblebee, lamentamos mucho lo ocurrido con su pedido. "
prompt = texto + "\n\nRespuesta del servicio al cliente:\n" + respuesta_inicial
outputs = generador(
    prompt,
    max_new_tokens=150,
    do_sample=True,
    temperature=1.0,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.3,
    eos_token_id=50256
)
print(outputs[0]['generated_text'])

## Principales aprendizajes y reflexión general

- **Clasificación de texto:** Permite detectar el tono emocional y automatizar tareas de análisis de satisfacción.
- **NER:** Extrae nombres, marcas y lugares para análisis estructurado.
- **Pregunta-respuesta:** Responde con precisión sobre el contenido del texto.
- **Resumen automático:** Resume textos extensos en segundos.
- **Traducción:** Traducción de calidad incluso en lenguaje coloquial.
- **Generación de texto:** Genera respuestas automáticas útiles en atención al cliente.

**Reflexión:** Estos modelos logran resultados impresionantes con poco código, aunque pueden fallar ante sarcasmo o lenguaje informal. Requieren supervisión humana en contextos críticos. Son aplicables en bots, análisis de feedback y monitoreo de redes sociales.